### Setting

In [ ]:
import os
import pandas as pd

import torch
import torch.nn as nn

#import gensim
#from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

import re
import numpy as np

from sklearn.preprocessing import LabelEncoder

import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

import torch.nn.functional as F

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

from collections import Counter

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available!")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU.")

GPU not available, using CPU.


# Data Loader

In [ ]:
I_plus_train = pd.read_parquet("I_plus_train.parquet")
I_minus_train = pd.read_parquet("I_minus_train.parquet")
I_plus_val = pd.read_parquet("I_plus_val.parquet")
I_minus_val = pd.read_parquet("I_minus_val.parquet")
test = pd.read_parquet("test.parquet")

In [ ]:
# Fit the scaler on the training data
scaler = StandardScaler()
scaler.fit(I_plus_train[['energy', 'key', 'loudness', 'mode', 'speechiness',
                         'acousticness', 'instrumentalness', 'liveness',
                         'valence', 'tempo']])

# Transform the data and replace the original columns with standardized values
I_plus_train[['energy', 'key', 'loudness', 'mode', 'speechiness',
              'acousticness', 'instrumentalness', 'liveness',
              'valence', 'tempo']] = scaler.transform(I_plus_train[['energy', 'key',
                                                                     'loudness', 'mode',
                                                                     'speechiness', 'acousticness',
                                                                     'instrumentalness', 'liveness',
                                                                     'valence', 'tempo']])

scaler.fit(I_minus_train[['energy', 'key', 'loudness', 'mode', 'speechiness',
                         'acousticness', 'instrumentalness', 'liveness',
                         'valence', 'tempo']])

# Transform the data and replace the original columns with standardized values
I_minus_train[['energy', 'key', 'loudness', 'mode', 'speechiness',
              'acousticness', 'instrumentalness', 'liveness',
              'valence', 'tempo']] = scaler.transform(I_minus_train[['energy', 'key',
                                                                     'loudness', 'mode',
                                                                     'speechiness', 'acousticness',
                                                                     'instrumentalness', 'liveness',
                                                                     'valence', 'tempo']])

#I_plus_train.iloc[:,6:16]

In [ ]:
# Fit the scaler on the training data
scaler.fit(I_plus_val[['energy', 'key', 'loudness', 'mode', 'speechiness',
                         'acousticness', 'instrumentalness', 'liveness',
                         'valence', 'tempo']])

# Transform the data and replace the original columns with standardized values
I_plus_val[['energy', 'key', 'loudness', 'mode', 'speechiness',
              'acousticness', 'instrumentalness', 'liveness',
              'valence', 'tempo']] = scaler.transform(I_plus_val[['energy', 'key',
                                                                     'loudness', 'mode',
                                                                     'speechiness', 'acousticness',
                                                                     'instrumentalness', 'liveness',
                                                                     'valence', 'tempo']])

scaler.fit(I_minus_val[['energy', 'key', 'loudness', 'mode', 'speechiness',
                         'acousticness', 'instrumentalness', 'liveness',
                         'valence', 'tempo']])

# Transform the data and replace the original columns with standardized values
I_minus_val[['energy', 'key', 'loudness', 'mode', 'speechiness',
              'acousticness', 'instrumentalness', 'liveness',
              'valence', 'tempo']] = scaler.transform(I_minus_val[['energy', 'key',
                                                                     'loudness', 'mode',
                                                                     'speechiness', 'acousticness',
                                                                     'instrumentalness', 'liveness',
                                                                     'valence', 'tempo']])


In [ ]:
I_plus_train

,pid,track_id,track_name,album_name,artist_name,danceability,energy,key,loudness,mode,...,lemmatized_album_name,track_sentences,album_sentences,artist_id,audio_features,track_embeddings,album_embeddings,new_pid,album_Bert_embedding,track_Bert_embedding
0,3,39055,Just Get High,Die Alone,Gazebos,0.337,0.230305,1.074302,0.056659,0.706732,...,die alone,"[get, high]","[die, alone]",7899,"[0.337, 0.701, 9.0, -6.974, 1.0, 0.0306, 0.001...","[[0.033203125, -0.08984375, -0.294921875, 0.11...","[[0.10009765625, 0.17578125, 0.043701171875, 0...",0,"[0.020719705, 0.25220913, -0.15911399, 0.18181...","[0.20254193, 0.32272324, -0.08004451, 0.049623..."
1,3,52439,Nothing's Gonna Hurt You Baby,I.,Cigarettes After Sex,0.509,-1.544450,-0.312059,-1.930558,0.706732,...,i,"[nothing, gonna, hurt, baby]",[i],4150,"[0.509, 0.331, 4.0, -14.083, 1.0, 0.0267, 0.27...","[[0.1640625, 0.056396484375, 0.08251953125, -0...","[[-0.2255859375, -0.01953125, 0.0908203125, 0....",0,"[-0.5305576, 0.11689519, -0.14607719, -0.25065...","[0.29745492, 0.15301126, -0.12392438, 0.069712..."
2,3,33924,I Like You,It's All In Your Head,dandelion hands,0.350,-2.652472,-0.312059,-3.162750,0.706732,...,head,[like],[head],24118,"[0.35, 0.1, 4.0, -18.491, 1.0, 0.0328, 0.989, ...","[[0.103515625, 0.1376953125, -0.00297546386718...","[[-0.0712890625, -0.07373046875, 0.19921875, -...",0,"[0.14113334, 0.32188103, 0.04410696, 0.1082610...","[0.015903875, 0.0660543, -0.09751289, 0.076490..."
3,3,15639,Crystal,Candy Apple Grey,Hüsker Dü,0.386,1.491819,0.519758,-1.155407,0.706732,...,candy apple grey,[crystal],"[candy, apple, grey]",9198,"[0.386, 0.964, 7.0, -11.31, 1.0, 0.076, 0.0589...","[[0.045654296875, -0.26953125, 0.0732421875, -...","[[-0.050048828125, -0.232421875, -0.0308837890...",0,"[-0.20137045, -0.049502436, -0.59205437, -0.02...","[-0.16305159, 0.15954661, 0.0659531, -0.022099..."
4,3,85903,You Say I'm in Love,You Say I'm in Love,Banes World,0.625,-1.765095,-1.143875,-2.658189,-1.414964,...,say im love,"[say, im, love]","[say, im, love]",1905,"[0.625, 0.285, 1.0, -16.686, 0.0, 0.0537, 0.43...","[[-0.0361328125, -0.12109375, 0.1337890625, 0....","[[-0.0361328125, -0.12109375, 0.1337890625, 0....",0,"[0.16846092, 0.17492926, -0.4919998, 0.0336023...","[0.16846092, 0.17492926, -0.4919998, 0.0336023..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7065,453,36170,If You're Gone,Mad Season,Matchbox Twenty,0.544,0.028846,1.074302,-0.004000,0.706732,...,mad season,"[youre, go]","[mad, season]",13932,"[0.544, 0.659, 9.0, -7.191, 1.0, 0.0298, 0.427...","[[0.28125, 0.033447265625, -0.035888671875, 0....","[[0.055419921875, -0.056640625, 0.150390625, 0...",100,"[-0.32580608, -0.16754074, -0.0823893, -0.0792...","[0.09254932, 0.23144999, -0.05158483, -0.12181..."
7066,453,6723,Barely Breathing,Duncan Sheik,Duncan Sheik,0.482,0.738748,-1.421147,0.056100,0.706732,...,duncan sheik,"[barely, breathing]","[duncan, sheik]",6250,"[0.482, 0.807, 0.0, -6.976, 1.0, 0.0457, 0.028...","[[0.2021484375, -0.25390625, -0.035400390625, ...","[[-0.06005859375, 0.1591796875, 0.019287109375...",100,"[-0.4858875, 0.24359304, -0.49777362, 0.172126...","[-0.34052244, 0.19653437, -0.12518123, -0.3739..."
7067,453,45970,Making Love Out of Nothing at All,The Best of Air Supply: Ones That You Love,Air Supply,0.386,0.230305,0.519758,0.330883,0.706732,...,best air supply one love,"[make, love, nothing]","[best, air, supply, one, love]",574,"[0.386, 0.701, 7.0, -5.993, 1.0, 0.0332, 0.14,...","[[-0.11328125, -0.036865234375, 0.09423828125,...","[[-0.126953125, 0.02197265625, 0.287109375, 0....",100,"[-0.1786602, 0.08967763, -0.31454346, 0.160023...","[-0.57437235, -0.37043536, -0.31799096, 0.5702..."
7068,453,32266,How Deep Is Your Love,Greatest,Bee Gees,0.633,-1.419737,-0.034786,-0.611990,-1.414964,...,great,"[deep, love]",[great],2071,"[0.633, 0.357, 5.0, -9.366, 0.0, 0.0264, 0.105...","[[-0.0361328125, 0.1494140625, 0.1376953125, 0...","[[0.07177734375, 0.2080078125, -0

In [ ]:
class MusicDataset(Dataset):
    def __init__(self, I_plus, I_minus):
        self.I_plus = I_plus
        self.I_minus = I_minus

    def __len__(self):
        return len(self.I_plus)

    def __getitem__(self, idx):
        I_plus_batch = self.I_plus.iloc[idx]
        I_minus_batch = self.I_minus.iloc[idx]


        I_plus_batch_tensor = {
            'pid': torch.tensor(I_plus_batch['new_pid'], dtype=torch.long),
            #'artist_id': torch.tensor(I_plus_batch['artist_id'], dtype=torch.long),
            'track_embeddings': torch.tensor(I_plus_batch['track_Bert_embedding'], dtype=torch.float32),
            'album_embeddings': torch.tensor(np.array(I_plus_batch['album_Bert_embedding']), dtype=torch.float32),
            'audio_features': torch.tensor(I_plus_batch['audio_features'], dtype=torch.float32)
        }

        I_minus_batch_tensor = {
            'pid': torch.tensor(I_minus_batch['new_pid'], dtype=torch.long),
            #'artist_id': torch.tensor(I_plus_batch['artist_id'], dtype=torch.long),
            'track_embeddings': torch.tensor(I_minus_batch['track_Bert_embedding'], dtype=torch.float32),
            'album_embeddings': torch.tensor(np.array(I_minus_batch['album_Bert_embedding']), dtype=torch.float32),
            'audio_features': torch.tensor(I_minus_batch['audio_features'], dtype=torch.float32)
        }

        return I_plus_batch_tensor, I_minus_batch_tensor


In [ ]:
trainSize = 70
valSize = 15

In [ ]:
# Create the Dataset
train_dataset = MusicDataset(I_plus_train, I_minus_train)
# Create the DataLoader
train_loader = DataLoader(train_dataset, batch_size=trainSize, shuffle=False)  # Modify batch size as needed

# Create the Dataset
val_dataset = MusicDataset(I_plus_val, I_minus_val)
# Create the DataLoader
val_loader = DataLoader(val_dataset, batch_size=valSize, shuffle=False)  # Modify batch size as needed

# Modelling

#### Autoencoder

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, embedding_dim=768, hidden_dim=256, bottleneck_dim=128):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


In [ ]:
class Album_Autoencoder(nn.Module):
    def __init__(self, embedding_dim=768, hidden_dim1=512, hidden_dim2=256, bottleneck_dim=128):
        super(Album_Autoencoder, self).__init__()

        # Encoder: 768 → 512 → 256 → 128
        self.encoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim1),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim1),

            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim2),

            nn.Linear(hidden_dim2, bottleneck_dim)
        )

        # Decoder: 128 → 256 → 512 → 768
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim2),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim2),

            nn.Linear(hidden_dim2, hidden_dim1),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim1),

            nn.Linear(hidden_dim1, embedding_dim)
        )

    def forward(self, x):
        z = self.encoder(x)       # Bottleneck representation (128)
        x_hat = self.decoder(z)   # Reconstructed 768-dim vector
        return x_hat, z


#### Album Embedding

In [ ]:
class Album_Embedding(nn.Module):
    def __init__(self):
        super(Album_Embedding, self).__init__()
        #self.autoencoder = Autoencoder()
        self.autoencoder = Album_Autoencoder()

    def forward(self, album_BERT):
        _, compressed = self.autoencoder(album_BERT)
        return compressed  # Shape: (batch, 128)

#### Track Embedding

In [ ]:
class Track_Embedding(nn.Module):
    def __init__(self):
        super(Track_Embedding, self).__init__()
        self.autoencoder = Autoencoder()

    def forward(self, track_BERT):
        _, compressed = self.autoencoder(track_BERT)
        return compressed  # Shape: (batch, 128)

### Audio Embedding


In [ ]:
#BEST AUDIO ONLY
class Audio_Embedding(nn.Module):
    def __init__(self, embedding_dim=11, unified_dim=128):
        super(Audio_Embedding, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.GELU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),

            nn.Linear(64, 128),
            nn.GELU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),

            nn.Linear(128, unified_dim),
            nn.GELU(),
            nn.BatchNorm1d(unified_dim)
        )

    def forward(self, audio_features):
        audio_embeds_unified = self.fc(audio_features)  # shape: (batch_size, 128)
        return audio_embeds_unified


#### Item Embedding

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=4, dropout=0.1):
        super(CrossAttention, self).__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value):
        attn_output, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(attn_output))

In [ ]:
# === Cross-Attention-Based Fusion (Takes embeddings as input) ===
class MultiFeatureCrossAttentionModel(nn.Module):
    def __init__(self, embed_dim=128, num_heads=4, out_dim=128):
        super(MultiFeatureCrossAttentionModel, self).__init__()

        self.cross_attn_album_track = CrossAttention(embed_dim, num_heads)
        self.cross_attn_audio_album_track = CrossAttention(embed_dim, num_heads)

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )

    def forward(self, audio_embed, album_embed, track_embed):
        album_track_attended = self.cross_attn_album_track(album_embed, track_embed, track_embed)
        audio_album_track_attended = self.cross_attn_audio_album_track(audio_embed, album_track_attended, album_track_attended)
        return self.mlp(audio_album_track_attended)  # Final item embedding


#audio track album

#audio and track - cross ateention /
# 2% -> 20%
# introduction, methodology introduction part (techniques) ->  Main chapter to result/ no need any reference.
# Result each technique
# structure sample (follow) /


In [ ]:
class Item_Embedding(nn.Module):
    def __init__(self, audio_embedding, album_embedding, track_embedding, cross_attention_model):
        super(Item_Embedding, self).__init__()
        self.audio_embedding = audio_embedding
        self.album_embedding = album_embedding
        self.track_embedding = track_embedding
        self.cross_attn = cross_attention_model

    def forward(self, audio_features, album_features, track_features):
        audio_embed = self.audio_embedding(audio_features)
        album_embed = self.album_embedding(album_features)
        track_embed = self.track_embedding(track_features)

        return self.cross_attn(audio_embed, album_embed, track_embed)

#### User Embedding

Over time, the user embedding should learn to represent the user’s preferences based on how often the user interacts with certain items.
This is because the model updates the user embedding based on the similarities between the user embedding and the item embeddings (which are influenced by item features).

In [ ]:
class User_Embedding(nn.Module):
    def __init__(self, num_playlists, embedding_dim=300, reduced_dim=128):
        super(User_Embedding, self).__init__()

        self.user_embedding = nn.Embedding(num_playlists, embedding_dim)


        self.fc = nn.Sequential(
          nn.Linear(embedding_dim,250),
          nn.ReLU(),
          nn.Linear(250, 200),
          nn.ReLU(),
          nn.Linear(200, 150),
          nn.ReLU(),
          nn.Linear(150, reduced_dim),
          nn.ReLU()
    )

        # Fully connected layer to reduce dimensionality to 128
        self.fc = nn.Linear(embedding_dim, reduced_dim)

    def forward(self, playlist_ids):
        user_embeds = self.user_embedding(playlist_ids)  # Shape: (batch_size, embedding_dim)
        user_embeds_reduced = self.fc(user_embeds)
        # Shape: (batch_size, reduced_dim]
        return user_embeds_reduced

#### User Item Embedding

In [ ]:
class UserItemEmbedding(nn.Module):
    def __init__(self, user_embedding_model, item_embedding_model):
        super(UserItemEmbedding, self).__init__()
        self.user_embedding = user_embedding_model
        self.item_embedding = item_embedding_model

    def forward(self, playlist_ids, audio_features, album_features, track_features):
        # Get user embeddings using playlist IDs
        user_embeds = self.user_embedding(playlist_ids)

        # Get item embeddings using all input features
        item_embeds = self.item_embedding(audio_features, album_features, track_features)
        return user_embeds, item_embeds

#### Model Initialisation

In [ ]:
# === Model Initialization ===
num_playlists = 101
embedding_dim = 128
reduced_dim = 128

audio_embedding_model = Audio_Embedding(embedding_dim=11, unified_dim=embedding_dim)
album_embedding_model = Album_Embedding()
track_embedding_model = Track_Embedding()

cross_attention_model = MultiFeatureCrossAttentionModel(
    embed_dim=embedding_dim,
    out_dim=reduced_dim
)

item_embedding_model = Item_Embedding(
    audio_embedding=audio_embedding_model,
    album_embedding=album_embedding_model,
    track_embedding=track_embedding_model,
    cross_attention_model=cross_attention_model
)

user_embedding_model = User_Embedding(
    num_playlists=num_playlists,
    embedding_dim=300,
    reduced_dim=reduced_dim
)

user_item_model = UserItemEmbedding(
    user_embedding_model=user_embedding_model,
    item_embedding_model=item_embedding_model
)


# Training

### Hyperparameters and Loss function

In [ ]:
class MaxMarginLoss(nn.Module):
    def __init__(self, margin):
        super(MaxMarginLoss, self).__init__()
        self.margin = margin

    def forward(self, user_embed, item_embed_pos, item_embed_neg):
        # # Calculate cosine similarities
        # pos_similarity = torch.cosine_similarity(user_embed, item_embed_pos)
        # neg_similarity = torch.cosine_similarity(user_embed, item_embed_neg)

        vx_pos = item_embed_pos - torch.mean(item_embed_pos)
        vx_neg = item_embed_neg - torch.mean(item_embed_neg)

        vy = user_embed - torch.mean(user_embed)

        # Pearson Similarity
        pos_similarity = torch.sum(vx_pos * vy) / (torch.sqrt(torch.sum(vx_pos ** 2)) * torch.sqrt(torch.sum(vy ** 2)))
        neg_similarity = torch.sum(vx_neg * vy) / (torch.sqrt(torch.sum(vx_neg ** 2)) * torch.sqrt(torch.sum(vy ** 2)))

        # Ensure margin is on the correct device
        margin_tensor = torch.tensor(self.margin, device=pos_similarity.device)


        # Max-margin hinge loss
        # Changed from 0 to torch.tensor(0., device=pos_similarity.device)
        loss = torch.max(torch.tensor(0., device=pos_similarity.device), margin_tensor - pos_similarity + neg_similarity)

        return loss

### Training

##### Validation function

In [ ]:
def validate(model, val_loader, loss_fn, recon_loss_fn):
    model.eval()
    loss_val = 0.0

    max_margin_loss = 0.0
    Track_recon_loss = 0.0
    Album_recon_loss = 0.0

    with torch.no_grad():
        for I_plus_batch, I_minus_batch in val_loader:

            # Move all tensors to device
            I_plus_batch = {key: value.to(device) for key, value in I_plus_batch.items()}
            I_minus_batch = {key: value.to(device) for key, value in I_minus_batch.items()}

            # === Positive samples ===
            user_embeds_pos = model.user_embedding(I_plus_batch['pid'])
            item_embeds_pos = model.item_embedding(
                audio_features=I_plus_batch['audio_features'],
                album_features=I_plus_batch['album_embeddings'],
                track_features=I_plus_batch['track_embeddings']
            )

            # === Negative samples ===
            item_embeds_neg = model.item_embedding(
                audio_features=I_minus_batch['audio_features'],
                album_features=I_minus_batch['album_embeddings'],
                track_features=I_minus_batch['track_embeddings']
            )

            # === Max-margin loss ===
            loss = loss_fn(user_embeds_pos, item_embeds_pos, item_embeds_neg)
            max_margin_loss += loss

            # === Final loss (e.g., reconstruction loss도 더하고 싶다면 여기에 추가 가능) ===
            total_loss = loss
            loss_val += total_loss.item()

    return loss_val / len(val_loader), max_margin_loss / len(val_loader)


##### Training Function

In [ ]:
music_pool = test
scaler = StandardScaler()
scaler.fit(music_pool[['energy', 'key', 'loudness', 'mode', 'speechiness',
                         'acousticness', 'instrumentalness', 'liveness',
                         'valence', 'tempo']])

music_pool[['energy', 'key', 'loudness', 'mode', 'speechiness',
              'acousticness', 'instrumentalness', 'liveness',
              'valence', 'tempo']] = scaler.transform(music_pool[['energy', 'key',
                                                                     'loudness', 'mode',
                                                                     'speechiness', 'acousticness',
                                                                     'instrumentalness', 'liveness',
                                                                     'valence', 'tempo']])

#Re-Assign the TrackID
unique_track_ids = music_pool['track_id'].unique()
track_id_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_track_ids)}
music_pool['new_track_id'] = music_pool['track_id'].map(track_id_mapping)
test['new_track_id'] = test['track_id'].map(track_id_mapping)



class MusicPoolDataset(Dataset):
    def __init__(self, I_plus):
        self.I_plus = I_plus


    def __len__(self):
        return len(self.I_plus)

    def __getitem__(self, idx):
        I_plus_batch = self.I_plus.iloc[idx]


        I_plus_batch_tensor = {
            #'pid': torch.tensor(I_plus_batch['new_pid'], dtype=torch.long),
            'track_id': torch.tensor(I_plus_batch['new_track_id'], dtype=torch.long),
            #'audio_features': torch.tensor(I_plus_batch['bert_embedding'], dtype=torch.float32),
            #'artist_id': torch.tensor(I_plus_batch['artist_id'], dtype=torch.long),
            'track_embeddings': torch.tensor(I_plus_batch['track_Bert_embedding'], dtype=torch.float32),
            'album_embeddings': torch.tensor(np.array(I_plus_batch['album_Bert_embedding']), dtype=torch.float32),
            'audio_features': torch.tensor(I_plus_batch['audio_features'], dtype=torch.float32)
        }

        return I_plus_batch_tensor

# Create the Dataset
music_pool_dataset = MusicPoolDataset(music_pool)
# Create the DataLoader
music_pool_loader = DataLoader(music_pool_dataset, batch_size=50, shuffle=False)  # Modify batch size as needed


In [ ]:
def calculate_item_embeddings(model, data_loader, device):
    all_item_embeddings = []
    all_track_ids = []

    model.eval()

    for batch_idx, I_plus_batch in enumerate(data_loader):
        # Move inputs to device
        audio_features = I_plus_batch['audio_features'].to(device)
        album_features = I_plus_batch['album_embeddings'].to(device)
        track_features = I_plus_batch['track_embeddings'].to(device)
        track_ids = I_plus_batch['track_id'].to(device)

        # Compute item embeddings using the full set of features
        item_embed = model.item_embedding(
            audio_features=audio_features,
            album_features=album_features,
            track_features=track_features
        )

        # Accumulate
        all_item_embeddings.append(item_embed)
        all_track_ids.append(track_ids)

    # Combine all batches
    all_item_embeddings = torch.cat(all_item_embeddings, dim=0).to(device)
    all_track_ids = torch.cat(all_track_ids, dim=0).to(device)

    return all_item_embeddings, all_track_ids


In [ ]:
def calculate_hit_ratio(model, all_user_embeddings, all_item_embeddings, track_ids, top_k):
  # Ensure embeddings are on the same device (CPU or GPU)
  all_user_embeddings = all_user_embeddings.to(all_item_embeddings.device)

  # Step 1: Center the embeddings by subtracting the mean
  user_means = all_user_embeddings.mean(dim=1, keepdim=True)
  item_means = all_item_embeddings.mean(dim=1, keepdim=True)

  all_user_embeddings_centered = all_user_embeddings - user_means
  all_item_embeddings_centered = all_item_embeddings - item_means

  # Step 2: Calculate the standard deviations
  user_std = all_user_embeddings_centered.std(dim=1, keepdim=True)
  item_std = all_item_embeddings_centered.std(dim=1, keepdim=True)

  # Step 3: Standardize the embeddings
  all_user_embeddings_standardized = all_user_embeddings_centered / user_std
  all_item_embeddings_standardized = all_item_embeddings_centered / item_std

  # Step 4: Compute Pearson correlation similarity
  # Reshape user embeddings for broadcasting (101, 30) -> (101, 1, 30)
  all_user_embeddings_standardized = all_user_embeddings_standardized.unsqueeze(1)

  # Calculate similarity: (101, 1, 30) * (1, 7602, 30) -> (101, 7602)
  #pearson_similarity = (all_user_embeddings_standardized * all_item_embeddings_standardized).sum(dim=2) / 30
  pearson_similarity = (all_user_embeddings_standardized * all_item_embeddings_standardized).sum(dim=2) / 128


  #top_k = 200
  top_k_tracks_ids = torch.topk(pearson_similarity, top_k, dim=1).indices

  # Count occurrences of each track in ground truth playlists
  ground_truth = test.groupby('new_pid')['new_track_id'].apply(list)
  hit_count = 0
  num_users = top_k_tracks_ids.shape[0]

  for user_id in range(num_users):
      recommended_tracks = {int(track.item()) for track in top_k_tracks_ids[user_id]}
      true_tracks = ground_truth.get(user_id, [])
      true_tracks_counter = Counter(true_tracks)  # Count duplicates in true_tracks

      # Calculate hit count considering duplicates
      for track in recommended_tracks:
          if track in true_tracks_counter:
              hit_count += true_tracks_counter[track]

  # Calculate hit rate

  hit_rate = hit_count * 100 / len(test)

  #print(f"Hit Count: {hit_count}")
  #print(f"Hit Rate: {hit_rate}%")

  return hit_count, hit_rate

In [ ]:
def training_loop(n_epochs, optimizer, model, loss_fn, recon_loss_fn, train_loader, val_loader, patience=3):
    train_losses = []
    val_losses = []
    train_maxmargin_losses = []
    val_maxmargin_losses = []
    hit_ratios = []
    hit_counts = []

    best_val_loss = float('inf')
    best_hit_ratio = 0.0
    best_model_state = None
    no_improvement_epochs = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        loss_train = 0.0
        loss_train_maxmargin = 0.0

        for I_plus_batch, I_minus_batch in tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}", unit="batch"):

            I_plus_batch = {key: value.to(device) for key, value in I_plus_batch.items()}
            I_minus_batch = {key: value.to(device) for key, value in I_minus_batch.items()}

            # Positive Samples
            user_embeds_pos = model.user_embedding(I_plus_batch['pid'])
            item_embeds_pos = model.item_embedding(
                audio_features=I_plus_batch['audio_features'],
                album_features=I_plus_batch['album_embeddings'],
                track_features=I_plus_batch['track_embeddings']
            )

            # Negative Samples
            item_embeds_neg = model.item_embedding(
                audio_features=I_minus_batch['audio_features'],
                album_features=I_minus_batch['album_embeddings'],
                track_features=I_minus_batch['track_embeddings']
            )

            # Loss
            loss = loss_fn(user_embeds_pos, item_embeds_pos, item_embeds_neg)
            loss_train_maxmargin += loss

            total_loss = loss
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            loss_train += total_loss.item()

        avg_train_loss = loss_train / len(train_loader)
        train_losses.append(avg_train_loss)
        train_maxmargin_losses.append(loss_train_maxmargin / len(train_loader))

        # === Validation ===
        loss_val, val_max_margin_loss = validate(model, val_loader, loss_fn, recon_loss_fn)
        val_losses.append(loss_val)
        val_maxmargin_losses.append(val_max_margin_loss)

        # === Evaluation Metrics ===
        all_user_embeddings = model.user_embedding(torch.arange(101).to(device))
        item_embeddings, track_ids = calculate_item_embeddings(model, music_pool_loader, device)
        hit_count, hit_rate = calculate_hit_ratio(model, all_user_embeddings, item_embeddings, track_ids, top_k=200)

        hit_ratios.append(hit_rate / 100)
        hit_counts.append(hit_count)

        print(f"Epoch {epoch}/{n_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {loss_val:.4f}, Hit Count: {hit_count}, Hit Ratio: {hit_rate:.2f}%")

        # === Best Model Saving by Hit Ratio ===
        if hit_rate > best_hit_ratio:
            best_hit_ratio = hit_rate
            best_model_state = model.state_dict()

        # === Early Stopping by Validation Loss ===
        if loss_val < best_val_loss:
            best_val_loss = loss_val
            no_improvement_epochs = 0
        else:
            no_improvement_epochs += 1

        if no_improvement_epochs >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    # Save best model by hit ratio
    torch.save(best_model_state, "best_model_by_hit_ratio.pt")
    print(f"Best model (Hit Ratio: {best_hit_ratio:.2f}%) saved.")

    return train_losses, val_losses, train_maxmargin_losses, val_maxmargin_losses, hit_counts, hit_ratios


#### training loop

##### 300dim user embedding with multiple layers + 1 layer item embedding, reduced dim = 30, lr =0.001

In [ ]:
user_item_model.to(device)

UserItemEmbedding(
  (user_embedding): User_Embedding(
    (user_embedding): Embedding(101, 300)
    (fc): Linear(in_features=300, out_features=128, bias=True)
  )
  (item_embedding): Item_Embedding(
    (audio_embedding): Audio_Embedding(
      (fc): Sequential(
        (0): Linear(in_features=11, out_features=64, bias=True)
        (1): GELU(approximate='none')
        (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (3): Dropout(p=0.2, inplace=False)
        (4): Linear(in_features=64, out_features=128, bias=True)
        (5): GELU(approximate='none')
        (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (7): Dropout(p=0.2, inplace=False)
        (8): Linear(in_features=128, out_features=128, bias=True)
        (9): GELU(approximate='none')
        (10): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (album_embedding): Album_Embedding(
     

In [ ]:
#Top 200 + 10k + Audio + Album + Track
train_losses, val_losses, train_maxmargin_losses, val_maxmargin_losses, hit_counts, hit_ratios = training_loop(
    n_epochs=500,  # Number of epochs
    optimizer=torch.optim.Adam(user_item_model.parameters(), lr=0.001),
    model=user_item_model,
    loss_fn= MaxMarginLoss(margin=0.2),
    recon_loss_fn=nn.MSELoss(),
    train_loader=train_loader,
    val_loader=val_loader,  # Validation loader
    patience=5  # Stop if no improvement for 5 epochs
)

# Evaluation

In [ ]:
# GPU Environment
loaded_model = user_item_model.to(device)
loaded_model.load_state_dict(torch.load(''))

#### Obtain All User Embeddings

In [ ]:
torch.arange(101)
numOfplaylists = 101
all_playlist_ids = torch.arange(numOfplaylists).to(device)
all_user_embeddings= loaded_model.user_embedding(all_playlist_ids)
print(all_user_embeddings.shape)  # shape: (#of playlists=101, 30)

#### Obtain Item Embedding for all songs in the music pool

In [ ]:
music_pool = pd.read_parquet('music_pool_10k_ver8.parquet')

scaler = StandardScaler()
scaler.fit(music_pool[['energy', 'key', 'loudness', 'mode', 'speechiness',
                         'acousticness', 'instrumentalness', 'liveness',
                         'valence', 'tempo']])

music_pool[['energy', 'key', 'loudness', 'mode', 'speechiness',
              'acousticness', 'instrumentalness', 'liveness',
              'valence', 'tempo']] = scaler.transform(music_pool[['energy', 'key',
                                                                     'loudness', 'mode',
                                                                     'speechiness', 'acousticness',
                                                                     'instrumentalness', 'liveness',
                                                                     'valence', 'tempo']])

class MusicPoolDataset(Dataset):
    def __init__(self, I_plus):
        self.I_plus = I_plus


    def __len__(self):
        return len(self.I_plus)

    def __getitem__(self, idx):
        I_plus_batch = self.I_plus.iloc[idx]

        I_plus_batch_tensor = {
            'track_id': torch.tensor(I_plus_batch['track_id'], dtype=torch.long),
            #'artist_id': torch.tensor(I_plus_batch['artist_id'], dtype=torch.long),
            #'track_embeddings': torch.tensor(np.array(I_plus_batch['track_embeddings']).tolist(), dtype=torch.float32),
            #'album_embeddings': torch.tensor(np.array(I_plus_batch['album_embeddings']).tolist(), dtype=torch.float32),
            'audio_features': torch.tensor(I_plus_batch['audio_features'].tolist(), dtype=torch.float32)
        }

        return I_plus_batch_tensor

# Create the Dataset
music_pool_dataset = MusicPoolDataset(music_pool)
# Create the DataLoader
music_pool_loader = DataLoader(music_pool_dataset, batch_size=50, shuffle=False)  # Modify batch size as needed

In [ ]:
def accumulate_item_embeddings(data_loader):
    all_item_embeddings = []  # list to store all embeddings
    all_track_ids = []  # list to store track ids

    for batch_idx, I_plus_batch in enumerate(data_loader):
        # Extract features from the I_plus batch
        audio_features = I_plus_batch['audio_features'].to(device)
        track_ids = I_plus_batch['track_id']  # Assuming track_id is in the batch data

        # Compute item embeddings for the batch
        item_concat = loaded_model.item_embedding(
            audio_features=audio_features
        )

        # Store the embeddings and track_ids for this batch
        all_item_embeddings.append(item_concat)
        all_track_ids.append(track_ids)

        #print(f"Batch {batch_idx} - Item embeddings shape: {item_concat.shape}")

    # Concatenate all embeddings and track_ids into single tensors
    all_item_embeddings = torch.cat(all_item_embeddings, dim=0)  # Concatenate along batch dim
    all_track_ids = torch.cat(all_track_ids, dim=0)  # Concatenate track ids

    return all_item_embeddings, all_track_ids


all_item_embeddings, all_track_ids = accumulate_item_embeddings(music_pool_loader)

In [ ]:
all_user_embeddings.shape
all_item_embeddings.shape

In [ ]:
# Ensure embeddings are on the same device (CPU or GPU)
all_user_embeddings = all_user_embeddings.to(all_item_embeddings.device)

# Step 1: Center the embeddings by subtracting the mean
user_means = all_user_embeddings.mean(dim=1, keepdim=True)
item_means = all_item_embeddings.mean(dim=1, keepdim=True)

all_user_embeddings_centered = all_user_embeddings - user_means
all_item_embeddings_centered = all_item_embeddings - item_means

# Step 2: Calculate the standard deviations
user_std = all_user_embeddings_centered.std(dim=1, keepdim=True)
item_std = all_item_embeddings_centered.std(dim=1, keepdim=True)

# Step 3: Standardize the embeddings
all_user_embeddings_standardized = all_user_embeddings_centered / user_std
all_item_embeddings_standardized = all_item_embeddings_centered / item_std

# Step 4: Compute Pearson correlation similarity
# Reshape user embeddings for broadcasting (101, 30) -> (101, 1, 30)
all_user_embeddings_standardized = all_user_embeddings_standardized.unsqueeze(1)

# Calculate similarity: (101, 1, 30) * (1, 7602, 30) -> (101, 7602)
pearson_similarity = (all_user_embeddings_standardized * all_item_embeddings_standardized).sum(dim=2) / 30

print(pearson_similarity.shape)  # Should be (101, 7602)
print(pearson_similarity)  # Pearson similarity between each user and each item

# Create a unique mapping for track_id to new_track_id
unique_track_ids = music_pool['track_id'].unique()
track_id_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_track_ids)}

# Apply this mapping to create a new column in music_pool
music_pool['new_track_id'] = music_pool['track_id'].map(track_id_mapping)
# music_pool

# Apply the same mapping to test data
test['new_track_id'] = test['track_id'].map(track_id_mapping)
test

#### Top k Calcuation

In [ ]:
from collections import Counter

top_k = 200
top_k_tracks_ids = torch.topk(pearson_similarity, top_k, dim=1).indices

# Count occurrences of each track in ground truth playlists
ground_truth = test.groupby('new_pid')['new_track_id'].apply(list)
hit_count = 0
num_users = top_k_tracks_ids.shape[0]

for user_id in range(num_users):
    recommended_tracks = {int(track.item()) for track in top_k_tracks_ids[user_id]}
    true_tracks = ground_truth.get(user_id, [])
    true_tracks_counter = Counter(true_tracks)  # Count duplicates in true_tracks

    # Calculate hit count considering duplicates
    for track in recommended_tracks:
        if track in true_tracks_counter:
            hit_count += true_tracks_counter[track]

# Calculate hit rate
print(f"Hit Count: {hit_count}")
hit_rate = hit_count * 100 / len(test)
print(f"Hit Rate: {hit_rate}%")
